# Contextual Clues Sampling (CCS, 2022)
---
[[paper]](https://arxiv.org/pdf/2210.07093)<br>CCS = Contextual Clues Sampling

Contextual Clues Sampling (CCS) — это метод сэмплирования негативных примеров, предложенный исследователями из Университета Вашингтона и Microsoft Research в 2022 году. Он предназначен для улучшения обучения моделей Dense Retrieval путем генерации более сложных, контекстно-чувствительных негативных примеров для контрастивного обучения.

### Контекст
Dense Retrieval модели, такие как DPR (2020), ANCE (2020), ColBERT (2020), использовали претренированных энкодерные модели, обучающиеся отличать  методом contrastive loss

Ключевой аспект контрастивного обучения — это качество **негативных примеров**. Модель учится различать релевантные (позитивные) и нерелевантные (негативные) документы для данного запроса. Если негативные примеры слишком просты, модель легко их отличает, и обучение малоэффективно. Нам нужны "сложные" негативные примеры (**hard negatives**), которые семантически близки к позитивным, но при этом нерелевантны запросу. Обучение на таких примерах заставляет модель изучать более тонкие различия.

### Проблема
Существующие методы генерации hard negatives имеют свои ограничения:
*   **In-batch negatives:** Самый простой метод, когда все другие документы в текущем батче используются как негативные для каждого запроса. Легко реализовать, но качество негативов сильно зависит от состава батча и часто бывает недостаточным.
*   **BM25 negatives:** Документы, которые BM25 высоко ранжирует для данного запроса, но которые не являются релевантными. Эти негативы часто эффективны, но они основаны на лексическом совпадении и могут не охватывать семантически сложные случаи.
*   **Approximate Nearest Neighbor (ANN) negatives:** Поиск негативных примеров в пространстве эмбеддингов, которые близки к позитивному документу или запросу, но не являются релевантными. Требует поддержания ANN-индекса всего корпуса, что может быть вычислительно дорого.
*   **Online hard negative mining (OHNM):** Постоянный пересчет самых "сложных" негативов во время обучения. Эффективно, но очень дорого с точки зрения вычислений, так как требует постоянного прохода по большому корпусу.

Основная проблема в том, что многие hard negatives, полученные лексическими или даже простыми семантическими методами, могут отличаться от позитивного документа только поверхностно. Модель должна научиться различать документы, которые не просто выглядят похоже, но и содержат **общие контекстуальные ключи**, но при этом приводят к разным ответам на запрос.

### Идея метода
Идея CCS заключается в том, чтобы генерировать hard negatives, которые разделяют **контекстуальные ключи** с позитивным документом, но при этом не являются релевантными запросу. Авторы предполагают, что если два документа разделяют общие контекстуальные элементы, но один из них является релевантным, а другой нет, то модель будет вынуждена учиться более тонким различиям для точной оценки релевантности.

Вместо того чтобы искать негативы, которые похожи на запрос (как BM25) или просто похожи на позитивный документ в пространстве эмбеддингов, CCS фокусируется на поиске негативов, которые **семантически связаны с *позитивным* документом**, но не отвечают на *запрос*. Это делает их "контекстно-сложными" (contextually hard).

### Постановка задачи
Решается задача **Dense Retrieval**: для данного запроса $Q$ необходимо найти $K$ наиболее релевантных текстовых документов $D = \{d_1, d_2, \ldots, d_N\}$ из большого корпуса. Модели обучаются посредством контрастивной функции потерь, где для пары $(Q, d^+)$ (запрос и релевантный документ) необходимо отличить $d^+$ от множества нерелевантных документов $d^-$.

### Существующие методы генерации негативных примеров
Как упомянуто выше, на момент появления CCS наиболее распространенными подходами были:
*   **In-batch negatives:** (часто используется в DPR (2020), ColBERT (2020)) — эффективны с точки зрения вычислений, но качество негативов непостоянно.
*   **BM25 hard negatives:** (используются в DPR (2020), ANCE (2020)) — показали хорошую эффективность, но ориентированы на лексическое совпадение.
*   **ANCE (2020)**: Асинхронное обновление индекса и майнинг hard negatives, близких к запросу в пространстве эмбеддингов, что требует значительных вычислительных ресурсов для поддержания актуального индекса.
*   **Gold-BM25 negatives:** Вариация BM25 hard negatives, где BM25 используется не только для ранжирования документов по запросу, но и для поиска документов, лексически похожих на *позитивный* документ. Это уже шаг в сторону "контекстуальных ключей", но CCS развивает эту идею глубже.

### Архитектура
CCS не вводит новой архитектуры модели. Это **методика обучения**, которая может быть применена к любой существующей архитектуре Dense Retrieval, использующей два энкодера (two-tower architecture), например, к DPR-подобным моделям на основе BERT.

Модель состоит из:
*   **Query Encoder:** Преобразует запрос в векторное представление (эмбеддинг).
*   **Passage Encoder:** Преобразует документ (пассаж) в векторное представление (эмбеддинг).
*   **Similarity Function:** Обычно скалярное произведение (dot-product) или косинусное сходство между эмбеддингами запроса и документа для вычисления оценки релевантности.

### Алгоритм обучения (с фокусом на сэмплирование)

Обучение Dense Retrieval модели с использованием CCS можно разбить на следующие шаги для каждой итерации:

1.  **Выбор батча:** Выбирается батч запросов $Q = \{q_1, \ldots, q_B\}$ и соответствующих им позитивных документов $D^+ = \{d_1^+, \ldots, d_B^+\}$.
2.  **Генерация Contextual Clue Negatives (CCN):**
    *   Для каждого позитивного документа $d_i^+$ в батче используется уже существующий (возможно, предварительно обученный) Dense Retrieval энкодер (или даже BM25), чтобы найти набор $K$ документов из всего корпуса, которые **наиболее похожи на $d_i^+$**. Эти документы формируют **кандидатский набор контекстуальных ключей**.
    *   Из этого кандидатского набора исключаются все документы, которые являются истинно релевантными для текущего запроса $q_i$.
    *   Оставшиеся документы являются **Contextual Clue Negatives (CCNs)**. Они семантически близки к позитивному документу $d_i^+$, но не релевантны запросу $q_i$, что делает их сложными для различения моделью.
3.  **Добавление других негативов:** К сгенерированным CCN могут быть добавлены другие типы негативов, например, in-batch negatives (все другие позитивные документы из текущего батча, которые не являются $d_i^+$) и/или BM25 hard negatives, для дальнейшего обогащения набора негативных примеров.
4.  **Кодирование:** Query Encoder кодирует все запросы из батча, Passage Encoder кодирует позитивные документы и все собранные негативные примеры.
5.  **Вычисление функции потерь:** Используется стандартная контрастивная функция потерь (например, Negative Log-Likelihood или Triplet Loss). Она максимизирует сходство между запросом и его позитивным документом, и минимизирует сходство между запросом и всеми негативными документами. Обучение на CCN заставляет модель учиться более тонким различиям, чтобы корректно ранжировать $d_i^+$ выше $CCN$.

**Ключевой момент:** Генерация CCN может выполняться оффлайн перед началом обучения или периодически обновляться во время обучения (подобно ANCE, но сфокусировано на связи с позитивным документом). В исходной статье авторы предлагают использовать "teacher" Dense Retriever для генерации этих ключей.

### Алгоритм инференса
Метод CCS никак не влияет на процесс инференса. После обучения модель Dense Retrieval работает стандартным образом:
1.  **Кодирование запроса:** Запрос $Q$ кодируется Query Encoder'ом, чтобы получить эмбеддинг $E_Q$.
2.  **Кодирование корпуса:** Все документы в корпусе предварительно кодируются Passage Encoder'ом, и их эмбеддинги $E_D = \{E_{d_1}, E_{d_2}, \ldots, E_{d_N}\}$ сохраняются в индексе ANN (Approximate Nearest Neighbor), таком как Faiss, HNSW и т.д.
3.  **Поиск:** Вычисляется сходство (например, скалярное произведение) между эмбеддингом запроса $E_Q$ и всеми эмбеддингами документов $E_D$ в индексе ANN.
4.  **Ранжирование:** Возвращаются $K$ документов с наивысшими оценками сходства.

### Результаты
CCS был протестирован на стандартных датасетах для Dense Retrieval, таких как **MS MARCO Passage Ranking**.
*   Модели, обученные с использованием CCS, **превзошли** базовые модели, использующие только in-batch negatives или BM25 hard negatives, по таким метрикам, как MRR@10 (Mean Reciprocal Rank at 10) и Recall@K.
*   Например, на MS MARCO CCS показал **улучшение до 1-2 процентных пункта MRR@10** по сравнению с сильными baseline-моделями, использующими продвинутые стратегии негативного сэмплирования. Это существенное улучшение в высококонкурентной области.
*   CCS продемонстрировал, что генерируемые им контекстно-чувствительные негативы действительно заставляют модель изучать более надежное и дискриминативное пространство эмбеддингов, улучшая способность модели различать тонкие семантические различия между релевантными и нерелевантными документами.
*   Преимущество CCS также заключается в его относительной простоте реализации по сравнению с более сложными схемами майнинга негативов, такими как ANCE, которые требуют постоянного обновления индекса.

## 📝 Критический анализ

# Contextual Clues Sampling (CCS, 2022)
---
[[paper]](https://arxiv.org/pdf/2203.07623)<br>CCS = Contextual Clues Sampling

Contextual Clues Sampling (CCS) — метод сэмплирования негативных примеров, предложенный исследователями из Университета Вашингтона и Microsoft Research в 2022 году. Он улучшает обучение моделей Dense Retrieval, создавая более сложные, контекстно-чувствительные негативные примеры для контрастивного обучения.

### Контекст
Dense Retrieval модели, такие как DPR (2020), ANCE (2020), ColBERT (2020), превосходят традиционные Sparse Retrieval подходы благодаря мощным языковым моделям и контрастивной функции потерь. Качество негативных примеров критично: сложные негативы (hard negatives) помогают модели различать тонкие семантические различия.

### Проблема
Существующие методы генерации hard negatives имеют ограничения:
- **In-batch negatives:** Просты, но зависят от состава батча.
- **BM25 negatives:** Эффективны, но основаны на лексическом совпадении.
- **ANN negatives:** Дороги в вычислениях.
- **OHNM:** Вычислительно затратны.

### Идея метода
CCS генерирует hard negatives, которые разделяют **контекстуальные ключи** с позитивным документом, но не релевантны запросу. Это заставляет модель учиться более тонким различиям.

### Постановка задачи
Решается задача **Dense Retrieval**: для запроса $Q$ найти $K$ наиболее релевантных документов из корпуса. Модели обучаются с контрастивной функцией потерь, различая позитивные и негативные документы.

### Архитектура
CCS — методика обучения, применимая к любой архитектуре Dense Retrieval с двумя энкодерами (two-tower architecture), например, DPR на основе BERT.

### Алгоритм обучения
1. **Выбор батча:** Выбирается батч запросов и позитивных документов.
2. **Генерация CCN:** Находятся документы, похожие на позитивный, но не релевантные запросу.
3. **Добавление других негативов:** Включаются in-batch и BM25 negatives.
4. **Кодирование:** Кодируются запросы и документы.
5. **Вычисление функции потерь:** Максимизируется сходство между запросом и позитивным документом, минимизируется с негативными.

### Алгоритм инференса
CCS не влияет на инференс. Запрос кодируется, документы предварительно кодируются и сохраняются в индексе ANN. Вычисляется сходство, возвращаются $K$ документов с наивысшими оценками.

### Результаты
CCS протестирован на MS MARCO Passage Ranking:
- Улучшение MRR@10 до 1-2 п.п. по сравнению с базовыми моделями.
- CCS улучшает способность модели различать тонкие семантические различия.
- Простота реализации по сравнению с ANCE.


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import torch

# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def encode_text(text):
    """Encodes text into BERT embeddings."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

# Example documents and queries
positive_doc = "The capital of France is Paris."
negative_doc_1 = "Paris is a city in France known for its art and culture."
negative_doc_2 = "The Eiffel Tower is a famous landmark in Paris."
query = "What is the capital of France?"

# Encode the query and documents
query_embedding = encode_text(query)
positive_embedding = encode_text(positive_doc)
negative_embedding_1 = encode_text(negative_doc_1)
negative_embedding_2 = encode_text(negative_doc_2)

# Calculate similarities
similarity_positive = cosine_similarity(query_embedding, positive_embedding)
similarity_negative_1 = cosine_similarity(query_embedding, negative_embedding_1)
similarity_negative_2 = cosine_similarity(query_embedding, negative_embedding_2)

print(f"Similarity with positive document: {similarity_positive[0][0]:.4f}")
print(f"Similarity with negative document 1: {similarity_negative_1[0][0]:.4f}")
print(f"Similarity with negative document 2: {similarity_negative_2[0][0]:.4f}")

# Contextual Clues Sampling (CCS) Example
# Assume we have a pre-trained dense retrieval model to find contextually similar documents
# Here we simulate this by using cosine similarity to find negatives similar to the positive doc

# Simulate finding contextually similar negatives
contextual_negatives = [negative_doc_1, negative_doc_2]
contextual_negatives_embeddings = [negative_embedding_1, negative_embedding_2]

# Calculate similarity between positive doc and potential negatives
similarities_to_positive = [cosine_similarity(positive_embedding, neg_emb)[0][0] for neg_emb in contextual_negatives_embeddings]

# Select the most contextually similar negative
most_contextual_negative_index = np.argmax(similarities_to_positive)
most_contextual_negative = contextual_negatives[most_contextual_negative_index]

print(f"Most contextually similar negative: {most_contextual_negative}")

# In a real CCS setup, this negative would be used in contrastive loss training
# to improve the model's ability to distinguish between subtle contextual differences.
```

### Explanation:

1. **BERT Encoding**: We use a pre-trained BERT model to encode text into embeddings. This is a common approach in dense retrieval systems to represent queries and documents.

2. **Similarity Calculation**: We calculate cosine similarity between the query and each document to simulate the retrieval process. The goal is to have the positive document (relevant to the query) have a higher similarity score than the negatives.

3. **Contextual Clues Sampling (CCS)**: We simulate the CCS process by finding negatives that are contextually similar to the positive document. In practice, a pre-trained dense retrieval model would be used to find these negatives.

4. **Selection of Contextual Negatives**: We calculate the similarity between the positive document and potential negatives, selecting the one most similar to the positive document. This negative is then used in training to improve the model's ability to distinguish subtle differences.

This example illustrates the core idea of CCS: generating hard negatives that share contextual clues with the positive document, forcing the model to learn finer distinctions.